# Notebook for practical exercises | Week #5 Lesson #3

## Introduction

This notebook contains practical hands-on exercises for the lesson about *Quantifying Uncertainty in AI Models*.

After this session, you will know how to evaluate and improve the performance of a classifier for a real-world segmentation problem:
- Load previously trained segmentation model.
- Perform Monte Carlo dropout inference and visualize uncertainty maps.
- Perform the calibration curve
- Compute ECE with M = 10 bins.
- Plot reliability diagrams before and after applying temperature scaling.
- Identify the 5 cases with the highest variance and discuss whether they correspond to difficult or ambiguous predictions.

## Dataset

We will use the [The Brain Resection Multimodal Imaging Database](https://www.cancerimagingarchive.net/collection/remind/) [1]. The Brain Resection Multimodal Imaging Database (ReMIND) contains pre- and intra-operative brain MRI collected on 114 consecutive patients who were surgically treated with image-guided tumor resection between 2018 and 2022.

[1] Juvekar, P., Dorent, R., Kögl, F., Torio, E., Barr, C., Rigolo, L., Galvin, C., Jowkar, N., Kazi, A., Haouchine, N., Cheema, H., Navab, N., Pieper, S., Wells, W. M., Bi, W. L., Golby, A., Frisken, S., & Kapur, T. (2023). The Brain Resection Multimodal Imaging Database (ReMIND) (Version 1) [dataset]. The Cancer Imaging Archive. [https://doi.org/10.7937/3RAG-D070](https://doi.org/10.7937/3RAG-D070)

Retrieve the repository

In [ ]:
!git clone https://github.com/albarqounilab/AIM.git

Import libraries

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
import sys
import warnings
warnings.filterwarnings(action="ignore")
os.environ["PYTHONWARNINGS"] = "ignore"


import argparse
import math
from glob import glob

!pip install -U segmentation-models-pytorch -q
import albumentations as A
import numpy as np
import pandas as pd
import SimpleITK as sitk
sitk.ProcessObject.SetGlobalWarningDisplay(False)
import torch
import torch.nn as nn
import torch.optim as optim
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import cv2

sitk.ProcessObject_SetGlobalWarningDisplay(False)

### Download

If needed, run the following cells to download and unzip the dataset. If you already have the dataset downloaded, comment the two lines.

<div class="alert alert-block alert-danger">
<b>Replace the <code>DATA_PATH</code> with the path where you want to store the data folder. By default, it will be stored at the root of this repository.</b> <br>
<b>If you have already downloaded the dataset, comment the following cell by adding a <code>#</code> before the <code>!</code></b>
</div>

In [ ]:
DATA_PATH = './'

# !curl https://uni-bonn.sciebo.de/s/s1t8QUZ02MF1Qoh/download --output {DATA_PATH}/data.zip
# !unzip {DATA_PATH}/data.zip -d {DATA_PATH}

### Dataset Class

Similarly to what we have done for previous weeks we to create a `Dataset` class for the segmentation task.
Observe the following cell and compare with the classes we created in lesson #1.

In [ ]:
# Load and observe available data
clinical_df = pd.read_csv(f'{DATA_PATH}/data/ReMIND/ReMIND_metadata.csv')
clinical_df.head() # Print the 5 fist rows of the dataframe

,Case Number,Age,Sex,Race,Ethnicity,Laterality,Previous Craniotomy,WHO Grade,Histopathology
0,1,41,Male,White,Not Hispanic,Right,No,3,Astrocytoma
1,2,22,Female,White,Not Hispanic,Right,No,2,Astrocytoma
2,3,30,Male,Asian,Not Hispanic,Left,No,2,Oligodendroglioma
3,4,23,Male,White,Hispanic,Left,No,1,Dysembryoplastic neuroepithelial tumor
4,5,27,Male,White,Not Hispanic,Right,No,Not assigned,Non-tumor epileptogenic brain parenchyma and g...


In [ ]:
#@title SegmentationDataset
class SegmentationDataset(Dataset):

    def __init__(self, clinical_df: pd.DataFrame, data_dir: str, size: int = 512, augment: bool = True):
        self.data_dir = data_dir
        self.size = size
        self.slices: list[tuple[np.ndarray, np.ndarray]] = []  # (img_slice, mask_slice)

        if augment:
            self.image_t = A.Compose(
                [
                    # A.Resize(size, size, interpolation=0),
                    # A.HorizontalFlip(),
                    # A.Normalize(mean=0.0, std=1.0),
                    # ToTensorV2(),
                    A.Resize(size, size, interpolation=0),
                    A.HorizontalFlip(p=0.5),
                    A.VerticalFlip(p=0.5),
                    A.ShiftScaleRotate(
                        shift_limit=0.0625, scale_limit=0.1, rotate_limit=15, p=0.5
                    ),
                    A.OneOf([
                        A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50),
                        A.GridDistortion(num_steps=5, distort_limit=0.3),
                        A.OpticalDistortion(distort_limit=0.05, shift_limit=0.05),
                    ], p=0.2),
                    # A.RandomBrightnessContrast(brightness_limit=0.2,
                    #                           contrast_limit=0.2, p=0.2),
                    A.RandomGamma(gamma_limit=(90, 110), p=0.1),
                    A.GaussNoise(var_limit=(0.05, 0.09), p=0.1),
                    A.CoarseDropout(max_holes=2,
                                  max_height=int(size * 0.15),
                                  max_width=int(size * 0.15),
                                  fill_value=0, p=0.5),
                    A.Normalize(mean=0.0, std=1.0),
                    ToTensorV2(),
                ]
            )
            self.mask_t = A.Compose(
                [
                    A.Resize(size, size, interpolation=0),
                    ToTensorV2(transpose_mask=True),
                ]
            )
        else:
            self.image_t = A.Compose([A.Resize(size, size, interpolation=0), A.Normalize(0.0, 1.0), ToTensorV2()])
            self.mask_t = A.Compose([A.Resize(size, size, interpolation=0), ToTensorV2(transpose_mask=True)])

        # Build slice index list once. We ignore patients without tumor mask
        for patient, _ in tqdm(
            zip(clinical_df["Case Number"], clinical_df["WHO Grade"]),
            total=len(clinical_df),
            desc="Indexing slices",
        ):
            idx = str(patient).zfill(3)
            img_dir = glob(f"{data_dir}/data/ReMIND/sub-{idx}/anat/T1w*")  # There might be *_T1w, *_T1ce, etc.
            seg_path = glob(f"{data_dir}/data/ReMIND/sub-{idx}/seg/ReMIND-{idx}-preop-SEG-tumor-*.nrrd")
            if not img_dir or not seg_path:
                continue

            # Load volumes once – keep in RAM temporarily.
            img_sitk = load_dicom_series_to_3d_image(img_dir[0])
            img_np = sitk.GetArrayFromImage(img_sitk)  # (Z, H, W)

            mask_sitk = sitk.ReadImage(seg_path[0])
            mask_sitk = reconstruct_mask_in_image_space(mask_sitk, img_sitk)
            mask_np = sitk.GetArrayFromImage(mask_sitk).astype(np.uint8)

            assert img_np.shape == mask_np.shape, "Image/mask dim mismatch"

            # For each axial slice, append if any tumor present.
            for k in range(img_np.shape[0]):
                if mask_np[k].any():
                    self.slices.append((img_np[k], mask_np[k]))

    def __len__(self):
        return len(self.slices)

    def __getitem__(self, idx: int):
        img_slice, mask_slice = self.slices[idx]
        transformed = self.image_t(image=img_slice, mask=mask_slice)
        img_t = transformed["image"].float()            # [1, H, W]
        mask_t = transformed["mask"].float()            # [H, W]
        mask_t = mask_t.unsqueeze(0)                    # now [1, H, W]
        return img_t, mask_t

# Helper functions to read the dicom and reconstruct the mask in image space
def load_dicom_series_to_3d_image(series_path: str) -> sitk.Image:
    """Read a DICOM series (folder full of slices) into a 3‑D SimpleITK image."""
    reader = sitk.ImageSeriesReader()
    dicom_names = reader.GetGDCMSeriesFileNames(series_path)
    reader.SetFileNames(dicom_names)
    img = reader.Execute()
    return img


def reconstruct_mask_in_image_space(mask_sitk: sitk.Image, reference_image: sitk.Image) -> sitk.Image:
    """Resample *mask_sitk* into the physical space & grid of *reference_image*."""
    return sitk.Resample(
        mask_sitk,
        reference_image,
        sitk.Transform(),
        sitk.sitkNearestNeighbor,
        0,
        mask_sitk.GetPixelID(),
    )

def inference(model, loader, device):
    model.eval()
    dices = []
    with torch.no_grad():
        for imgs, masks in tqdm(loader, desc="Inference", leave=False):
            imgs, masks = imgs.to(device), masks.to(device)
            preds = model(imgs)
            dices.append(dice_coeff(preds.cpu(), masks.cpu()))
    return np.mean(dices)


def dice_coeff(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-6):
    pred = torch.sigmoid(pred) > 0.5
    intersection = (pred & (target > 0.5)).float().sum(dim=(1, 2, 3))
    union = pred.float().sum(dim=(1, 2, 3)) + target.float().sum(dim=(1, 2, 3))
    dice = (2.0 * intersection + eps) / (union + eps)
    return dice.mean().item()

#perform splitting
patient_list = clinical_df['Case Number'].unique()

patient_train, patient_test = train_test_split(
    patient_list, # List or array to split
    test_size= 0.2, # Size of the subset
    random_state=42)

patient_train, patient_val = train_test_split(
    patient_train, # List or array to split
    test_size= 0.1, # Size of the subset
    random_state=42)

train_df = clinical_df.loc[clinical_df['Case Number'].isin(patient_train)]
test_df = clinical_df.loc[clinical_df['Case Number'].isin(patient_test)]
val_df = clinical_df.loc[clinical_df['Case Number'].isin(patient_val)]


test_dataset = SegmentationDataset(test_df, DATA_PATH)

Indexing slices: 100%|██████████████████████████| 23/23 [00:05<00:00,  4.17it/s]


## Load segmentation model

In [ ]:
import segmentation_models_pytorch as smp
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

model = smp.Unet(encoder_name="resnet50", encoder_weights="imagenet", in_channels=1, classes=1) # Binary class
model.to(device)

checkpoint_path = "./checkpoint/fcn_segmentation010_dice0.6615.pt" # Replace with the actual path to your saved checkpoint
checkpoint_path = "./checkpoint/fcn_segmentation020_dice0.6266.pt" # Replace with the actual path to your saved checkpoint

checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])

model.eval()
test_dataloader = DataLoader(test_dataset, batch_size=1, num_workers=8, pin_memory=True)

inference_dice = inference(model, test_dataloader, device)
print('DICE score:', inference_dice)

DICE score: 0.1431776089666604


##  Monte Carlo Dropout Inference

we explore how to quantify uncertainty in brain tumor segmentation using Monte Carlo (MC) Dropout. Understanding uncertainty helps clinicians interpret model confidence and identify cases that need further review.

We will we apply MC Dropout by performing multiple stochastic forward passes with dropout enabled. This produces a distribution of predictions from which we estimate the mean prediction and the uncertainty (variance).

In [ ]:
def mc_dropout_predict(model, input_tensor, T=30):
    model.train()
    preds = []
    with torch.no_grad():
        for _ in range(T):
            output = model(input_tensor)
            preds.append(torch.softmax(output, dim=1))
    preds = torch.stack(preds)
    mean_pred = preds.mean(dim=0)
    var_pred = preds.var(dim=0)
    return mean_pred, var_pred


<div class="alert alert-block alert-info">
<b>Q1.</b>  Why is uncertainty estimation especially important in medical image segmentation compared to standard classification tasks?
</div>

##  Visualize Uncertainty Maps

We visualize the mean segmentation alongside the uncertainty map. High uncertainty often highlights ambiguous or poorly defined tumor boundaries, guiding human oversight.

In [ ]:
import matplotlib.pyplot as plt

def plot_segmentation_with_uncertainty(input_img, mean_pred, var_pred, slice_idx=64):
    plt.figure(figsize=(18,5))

    # Original image
    plt.subplot(1, 3, 1)
    plt.imshow(input_img[0, 0, :, :, slice_idx].cpu(), cmap='gray')
    plt.title('Input MRI Slice'); plt.axis('off')

    # Mean prediction
    plt.subplot(1, 3, 2)
    plt.imshow(mean_pred[0, 1, :, :, slice_idx].cpu(), cmap='gray')
    plt.title('Mean Segmentation'); plt.axis('off')

    # Uncertainty map
    plt.subplot(1, 3, 3)
    plt.imshow(var_pred[0, 1, :, :, slice_idx].cpu(), cmap='hot')
    plt.title('Uncertainty Map'); plt.axis('off')

    plt.tight_layout()
    plt.show()


<div class="alert alert-block alert-info">
<b>Q2.</b> What does a high variance across predictions tell us about the model’s knowledge in that region?
</div>

##  Calibration Curve and Metrics

Now we'll use calibration curves to assess how well the model's predicted probabilities reflect actual correctness. This complements uncertainty maps by checking confidence reliability.

In [ ]:

def plot_calibration_curve(pred_probs, true_labels, n_bins=10, title="Calibration Curve"):

    prob_true, prob_pred = calibration_curve(true_labels, pred_probs, n_bins=n_bins, strategy='uniform')


    bin_totals = np.histogram(pred_probs, bins=n_bins, range=(0, 1))[0]
    acc_per_bin = prob_true
    conf_per_bin = prob_pred
    ece = np.sum((np.abs(acc_per_bin - conf_per_bin)) * (bin_totals / bin_totals.sum()))


    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6, 8), gridspec_kw={'height_ratios': [3, 1]})


    ax1.plot(prob_pred, prob_true, "o-", label="Model")
    ax1.plot([0, 1], [0, 1], "--", color="gray", label="Perfect Calibration")
    ax1.set_xlabel("Predicted Probability")
    ax1.set_ylabel("True Frequency")
    ax1.set_title(f"{title} (ECE = {ece:.3f})")
    ax1.legend(loc="upper left")
    ax1.set_xlim(0, 1)
    ax1.set_ylim(0, 1)

    # Histogram
    bin_centers = (np.linspace(0, 1, n_bins + 1)[:-1] + np.linspace(0, 1, n_bins + 1)[1:]) / 2
    ax2.hist(pred_probs, bins=n_bins, range=(0, 1), color="C0", edgecolor="black")
    ax2.set_xlabel("Predicted Probability")
    ax2.set_ylabel("Pixel Count")

    plt.tight_layout()
    plt.show()

    return ece


Indexing slices: 100%|████████████████████████| 114/114 [00:39<00:00,  2.87it/s]


<div class="alert alert-block alert-info">
<b>Q3.</b> In your visualization, do regions of high uncertainty correspond to anatomically complex areas or data boundaries?
</div>

Think of a clinical setting:
If you were a radiologist using this segmentation model, how would you use the uncertainty map in your decision-making? Can you identify a case where high uncertainty could change the course of treatment?